# 51job 职位列表采集（Selenium 版）

## 断点续采机制
1. 可随时手动停止（中断内核 ■）
2. 每采完一页自动保存到 checkpoint.csv 和 状态.txt
3. 再次运行自动从断点页继续
4. 已采集的「标题链接」自动跳过
5. 没有下一页时自动终止

## 生成文件（桌面）
| 文件 | 说明 |
|------|------|
| `51job_列表采集结果.csv` | 最终数据 |
| `51job_列表采集结果_checkpoint.csv` | 断点用，不要删 |
| `51job_列表采集状态.txt` | 断点用，不要删 |


## Cell 1 — 安装依赖

In [ ]:
!pip install selenium webdriver-manager pandas beautifulsoup4 lxml -q
print('安装完成 ✓')

## Cell 2 — 配置区域（只改这里）

In [1]:
import os

# ══ 把浏览器地址栏里搜索结果页的完整 URL 粘贴到这里 ══
SEARCH_URL = 'https://we.51job.com/pc/search?searchType=2&function=7501&jobArea=080400,010000,020000,030200,040000&sortType=0&pageNum=1'

MAX_PAGES       = 0    # 0 = 不限制
PAGE_LOAD_WAIT  = 8    # 首次加载等待秒数（网速慢可调大）
SCROLL_WAIT     = 3    # 每次滚动/翻页后等待秒数

DESKTOP        = os.path.join(os.path.expanduser('~'), 'Desktop')
OUTPUT_CSV     = os.path.join(DESKTOP, '51job_列表采集结果.csv')
CHECKPOINT_CSV = os.path.join(DESKTOP, '51job_列表采集结果_checkpoint.csv')
STATE_FILE     = os.path.join(DESKTOP, '51job_列表采集状态.txt')

print('配置加载完成 ✓')

配置加载完成 ✓


## Cell 3 — 工具函数

In [2]:
import json, time, random, re
from datetime import datetime
import pandas as pd
from bs4 import BeautifulSoup

def load_state():
    if os.path.exists(STATE_FILE):
        try:
            with open(STATE_FILE, 'r', encoding='utf-8') as f:
                return json.loads(f.read())
        except: pass
    return {'next_page': 1, 'collected': 0}

def save_state(next_page, collected):
    with open(STATE_FILE, 'w', encoding='utf-8') as f:
        f.write(json.dumps({'next_page': next_page, 'collected': collected,
            'updated_at': datetime.now().strftime('%Y-%m-%d %H:%M:%S')}, ensure_ascii=False))

def load_done_links():
    if os.path.exists(CHECKPOINT_CSV):
        try:
            df = pd.read_csv(CHECKPOINT_CSV, encoding='utf-8-sig', usecols=['标题链接'])
            return set(df['标题链接'].dropna().tolist())
        except: pass
    return set()

def append_rows(rows, filepath, write_header):
    pd.DataFrame(rows).to_csv(filepath, mode='a', header=write_header,
                              index=False, encoding='utf-8-sig')

print('工具函数定义完成 ✓')

工具函数定义完成 ✓


## Cell 4 — 浏览器启动函数

In [3]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

def create_driver():
    opts = Options()
    opts.add_argument('--no-sandbox')
    opts.add_argument('--disable-dev-shm-usage')
    opts.add_argument('--disable-blink-features=AutomationControlled')
    opts.add_experimental_option('excludeSwitches', ['enable-automation'])
    opts.add_experimental_option('useAutomationExtension', False)
    opts.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
        'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36')
    # 开启性能日志，用于拦截 API JSON
    opts.set_capability('goog:loggingPrefs', {'performance': 'ALL'})
    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=opts)
    # 隐藏 webdriver 特征
    driver.execute_cdp_cmd('Page.addScriptToEvaluateOnNewDocument',
        {'source': 'Object.defineProperty(navigator,"webdriver",{get:()=>undefined})'})
    return driver

print('浏览器函数定义完成 ✓')

浏览器函数定义完成 ✓


## Cell 5 — 页面数据解析函数

In [4]:
def parse_page(driver, done_links):
    """
    双通道解析：
    通道1：从浏览器性能日志拦截 search-pc API 的 JSON 响应（最准确）
    通道2：直接解析渲染后的 HTML DOM（API 日志捞不到时的备用）
    两个通道都没数据才返回空列表。
    """
    crawl_time = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    jobs = []

    # ── 通道1：API JSON ────────────────────────────────────────────
    try:
        logs = driver.get_log('performance')
        for entry in logs:
            try:
                msg = json.loads(entry['message'])['message']
                if msg.get('method') != 'Network.responseReceived': continue
                url = msg['params']['response']['url']
                if 'search-pc' not in url: continue
                req_id = msg['params']['requestId']
                body_raw = driver.execute_cdp_cmd('Network.getResponseBody', {'requestId': req_id})
                data = json.loads(body_raw.get('body', '{}'))
                items = data.get('resultbody', {}).get('job', {}).get('items', [])
                for item in items:
                    link = item.get('job_href','') or item.get('jobHref','')
                    if not link:
                        jid = item.get('jobid','') or item.get('job_id','')
                        link = f'https://we.51job.com/pc/rec?jobid={jid}' if jid else ''
                    if link and link in done_links: continue
                    ind = item.get('jobIndustry','') or item.get('industryField','')
                    if isinstance(ind, list): ind = '，'.join(str(i) for i in ind)
                    jobs.append({'标题': item.get('jobName',''),
                        '标题链接': link,
                        'sal':  item.get('provideSalaryString','') or item.get('salary',''),
                        'er':   item.get('companyName',''),
                        'er2':  str(ind),
                        'er1':  item.get('companyType','') or item.get('companyTypeString',''),
                        '字段1': item.get('companySize','') or item.get('companySizeString',''),
                        '采集时间': item.get('issueDate',''),
                        '抓取时间': crawl_time})
                    if link: done_links.add(link)
            except: continue
    except: pass

    if jobs:
        return jobs  # 通道1成功，直接返回

    # ── 通道2：DOM 解析（自适应选择器）────────────────────────────
    soup = BeautifulSoup(driver.page_source, 'lxml')

    # 51job 职位卡片：尝试多种选择器，用第一个命中的
    cards = []
    for sel in [
        'div.e',                      # 旧版
        'div[class*="joblist"] div[class*="item"]',  # 新版可能的class
        'div.j_joblist > div',        # 另一种结构
        'ul.j_result_list > li',      # 列表结构
    ]:
        cards = soup.select(sel)
        if cards: break

    for card in cards:
        try:
            # 标题 & 链接：找卡片里第一个有 href 的 <a>
            a = card.select_one('a[href*="51job"]') or card.select_one('a[href]')
            if not a: continue
            title = a.get_text(strip=True)
            link  = a.get('href', '')
            if not title or not link: continue
            if link in done_links: continue

            # 薪资：找包含 k/千/万/元 的文本节点
            sal = ''
            for t in card.find_all(string=re.compile(r'[kK千万]|\d+-\d+')):
                sal = t.strip(); break

            # 公司名：找第二个 <a> 或带 company/corp 关键字的元素
            anchors = card.select('a')
            company = anchors[1].get_text(strip=True) if len(anchors) > 1 else ''

            # 其他标签（行业/性质/规模）
            tags = [t.get_text(strip=True) for t in card.select('em, span[class*="tag"]') if t.get_text(strip=True)]

            # 时间
            time_tag = card.select_one('span[class*="time"], p[class*="time"]')
            post_time = time_tag.get_text(strip=True) if time_tag else ''

            jobs.append({'标题': title, '标题链接': link,
                'sal': sal, 'er': company,
                'er2': tags[0] if len(tags)>0 else '',
                'er1': tags[1] if len(tags)>1 else '',
                '字段1': tags[2] if len(tags)>2 else '',
                '采集时间': post_time, '抓取时间': crawl_time})
            done_links.add(link)
        except: continue

    return jobs


def wait_jobs(driver, timeout=12):
    """
    等待职位内容出现。
    51job 页面结构会变，这里用最宽泛的方式判断：
    只要页面有包含薪资关键词（k/千/万）的文本就认为加载完成。
    """
    deadline = time.time() + timeout
    while time.time() < deadline:
        src = driver.page_source
        # 页面里出现薪资/职位相关词就算加载完成
        if any(kw in src for kw in ['provideSalary', '万/月', '千/月', 'jobName', 'companyName']):
            return True
        time.sleep(1)
    return False


def goto_next_page(driver):
    """
    点击下一页。
    优先用 API 日志里的页码参数自己构造 URL 跳转，
    避免因找不到按钮而失败。
    返回 True=成功, False=已是最后一页。
    """
    # 先尝试找按钮
    for sel in [
        'button.btn-next:not([disabled])',
        'a.next:not(.disabled)',
        'li.next:not(.disabled) a',
        'a[aria-label*="下一页"]:not([aria-disabled="true"])',
        '[class*="next"]:not([disabled]):not([class*="disabled"])',
    ]:
        try:
            btns = driver.find_elements(By.CSS_SELECTOR, sel)
            if btns:
                driver.execute_script('arguments[0].click();', btns[0])
                return True
        except: continue
    return False

print('解析函数定义完成 ✓')

解析函数定义完成 ✓


## Cell 6 — 主采集循环
运行后会弹出 Chrome 窗口自动采集，随时中断（■）进度自动保存。
**不要手动关闭弹出的 Chrome 窗口**，中断请用 Jupyter 的 ■ 按钮。


In [5]:
# ── 加载进度 ────────────────────────────────────────────────────────
state      = load_state()
start_page = state['next_page']
total_col  = state['collected']
done_links = load_done_links()
print(f'接续进度：从第 {start_page} 页开始，已采 {total_col} 条')

out_hdr  = not (os.path.exists(OUTPUT_CSV)     and os.path.getsize(OUTPUT_CSV)     > 0)
ckpt_hdr = not (os.path.exists(CHECKPOINT_CSV) and os.path.getsize(CHECKPOINT_CSV) > 0)

# ── 启动浏览器 ──────────────────────────────────────────────────────
print('启动 Chrome……')
driver = create_driver()
page = start_page

try:
    # 打开目标页（续采时直接跳到断点页）
    url = SEARCH_URL
    if start_page > 1:
        sep = '&' if '?' in SEARCH_URL else '?'
        url = f'{SEARCH_URL}{sep}pageNum={start_page}'
    print(f'打开: {url[:80]}...')
    driver.get(url)
    print(f'等待页面加载 {PAGE_LOAD_WAIT}s……')
    time.sleep(PAGE_LOAD_WAIT)

    while True:
        if MAX_PAGES > 0 and page > MAX_PAGES:
            print(f'已达最大页数 {MAX_PAGES}，自动终止。')
            break

        print(f'[第 {page:>4} 页] 等待数据……', end='', flush=True)

        # 等待页面数据就绪（宽泛判断，不依赖特定 DOM class）
        loaded = wait_jobs(driver, timeout=15)
        if not loaded:
            # 超时但不卡死：继续尝试解析，也许有部分数据
            print(' 等待超时，尝试直接解析……', end='', flush=True)

        time.sleep(SCROLL_WAIT)  # 再等一会儿让 API 日志到达

        rows = parse_page(driver, done_links)

        if not rows:
            # 真的没数据才视为最后一页，不弹 input()
            print(' 无数据，采集完毕。')
            # 如果是第一页就没数据，打印页面源码片段帮助诊断
            if page == start_page:
                src = driver.page_source[:600]
                print(f'\n[诊断] 页面源码前600字符:\n{src}')
            break

        # 写入 CSV + 保存断点
        append_rows(rows, OUTPUT_CSV,     out_hdr)
        append_rows(rows, CHECKPOINT_CSV, ckpt_hdr)
        out_hdr  = False
        ckpt_hdr = False
        total_col += len(rows)
        save_state(page + 1, total_col)

        print(f' {len(rows):>2} 条，累计 {total_col} 条')

        # 翻页
        if not goto_next_page(driver):
            print('已是最后一页，采集完毕。')
            break

        page += 1
        time.sleep(SCROLL_WAIT)

except KeyboardInterrupt:
    print(f'\n[已停止] 进度已保存，下次从第 {page} 页继续')
finally:
    try: driver.quit()
    except: pass
    print(f'\n浏览器已关闭。共采集 {total_col} 条 → {OUTPUT_CSV}')

接续进度：从第 4 页开始，已采 60 条
启动 Chrome……
打开: https://we.51job.com/pc/search?searchType=2&function=7501&jobArea=080400,010000,...
等待页面加载 8s……
[第    4 页] 等待数据…… 等待超时，尝试直接解析…… 17 条，累计 77 条
[第    5 页] 等待数据…… 等待超时，尝试直接解析…… 20 条，累计 97 条
[第    6 页] 等待数据…… 等待超时，尝试直接解析…… 19 条，累计 116 条
[第    7 页] 等待数据…… 等待超时，尝试直接解析…… 20 条，累计 136 条
[第    8 页] 等待数据…… 等待超时，尝试直接解析…… 20 条，累计 156 条
[第    9 页] 等待数据…… 等待超时，尝试直接解析…… 20 条，累计 176 条
[第   10 页] 等待数据…… 等待超时，尝试直接解析…… 20 条，累计 196 条
[第   11 页] 等待数据…… 等待超时，尝试直接解析…… 20 条，累计 216 条
[第   12 页] 等待数据…… 等待超时，尝试直接解析…… 20 条，累计 236 条
[第   13 页] 等待数据…… 等待超时，尝试直接解析…… 20 条，累计 256 条
[第   14 页] 等待数据…… 等待超时，尝试直接解析…… 20 条，累计 276 条
[第   15 页] 等待数据…… 等待超时，尝试直接解析…… 20 条，累计 296 条
[第   16 页] 等待数据…… 等待超时，尝试直接解析…… 20 条，累计 316 条
[第   17 页] 等待数据…… 等待超时，尝试直接解析…… 20 条，累计 336 条
[第   18 页] 等待数据…… 等待超时，尝试直接解析…… 20 条，累计 356 条
[第   19 页] 等待数据…… 等待超时，尝试直接解析…… 20 条，累计 376 条
[第   20 页] 等待数据…… 等待超时，尝试直接解析…… 20 条，累计 396 条
[第   21 页] 等待数据…… 等待超时，尝试直接解析…… 20 条，累计 416 条
[第   22 页] 等待数据…… 等待超时，尝试直接解析…… 20 条，累计 

## Cell 7 — 结果预览

In [6]:
df = pd.read_csv(OUTPUT_CSV, encoding='utf-8-sig')
print(f'共 {len(df)} 条，列：{list(df.columns)}')
df.head(5)

共 853 条，列：['标题', '标题链接', 'sal', 'er', 'er2', 'er1', '字段1', '采集时间', '抓取时间']


,标题,标题链接,sal,er,er2,er1,字段1,采集时间,抓取时间
0,Staff Quality Engineer,http://te.51job.com/show_job_detail.html?jobid...,NaN,泰科电子（上海）,NaN,外资（欧美）,10000人以上,NaN,2026-05-12 18:49:19
1,KD项目包装数据库专员,https://jobs.51job.com/shanghai-ypq/171924257....,1-1.3万·14薪,安吉天地物流科技,NaN,民营,少于50人,NaN,2026-05-12 18:49:19
2,数据分析师,https://jobs.51job.com/shanghai-pdxq/171811813...,1-1.5万·13薪,上海学获帮教育科技,NaN,民营,NaN,NaN,2026-05-12 18:49:19
3,数据分析师,https://jobs.51job.com/shanghai/170740993.html...,9千-1.3万,上海塑盛电子商务,NaN,民营,少于50人,NaN,2026-05-12 18:49:19
4,优化工程师（数据分析方向）,https://jobs.51job.com/shanghai/156229312.html...,1.3-2.5万,上海吉祥航空,NaN,民营,5000-10000人,NaN,2026-05-12 18:49:19


## Cell 8 — 诊断工具（采不到数据时用）
如果 Cell 6 显示「无数据」，运行此 Cell 分析页面结构，帮助定位问题。


In [7]:
# 单独打开一个浏览器，查看实际页面结构
diag_driver = create_driver()
diag_driver.get(SEARCH_URL)
time.sleep(PAGE_LOAD_WAIT)

src = diag_driver.page_source
soup_d = BeautifulSoup(src, 'lxml')

print('=== 页面 <div> class 列表（前30个）===')
divs = soup_d.find_all('div', class_=True)
classes = list(dict.fromkeys(  # 去重保序
    ' '.join(d.get('class',[])) for d in divs if d.get('class')
))[:30]
for c in classes:
    print(' ', c)

print('\n=== <a> 标签样本（含 href 的前10个）===')
for a in soup_d.select('a[href]')[:10]:
    print(f'  text={a.get_text(strip=True)[:20]!r}  href={a["href"][:60]}')

print('\n=== 页面是否含关键词 ===')
for kw in ['jobName','companyName','provideSalary','search-pc','resultbody']:
    print(f'  {kw}: {kw in src}')

diag_driver.quit()
print('\n诊断完成，根据上面的 class 名可以修正选择器')

=== 页面 <div> class 列表（前30个）===
  index
  app-header-wrap
  header-warning
  warning-text
  close-btn
  app-header
  header-container
  header-left
  logo
  menu
  menu-item
  menu-item active
  el-popover el-popper header-popover
  popover-menu
  header-right
  right-menu
  item
  el-popover el-popper download-popover
  popover-container
  code-wrap
  download el-popover__reference
  post search-bg
  search-select
  j_search
  in
  search_container shake-animation
  search-wrap
  search-input e_ip
  search-btn
  history-wrap

=== <a> 标签样本（含 href 的前10个）===
  text=''  href=https://www.51job.com
  text='首页'  href=https://www.51job.com
  text='搜索'  href=javascript:;
  text='校园招聘'  href=https://xy.51job.com/default-xs.php
  text='测培商城'  href=https://td.51job.com/home?jobsign=1&partner=dlhyyx_cpdy_51jo
  text='典范雇主'  href=https://hrawards.51job.com/
  text='职场资讯'  href=https://mkt.51job.com/careerpost/default_res.php
  text='企业培训'  href=https://tr.51job.com/default_train.php
  text='职场百科'  h